# Unified MCTS — self-play collect (one shard)

Play **unified vs Stockfish** and write a memmap shard:

- `sp_shard_{SHARD_ID}/value_ft/` — error-mined STM value labels
- `sp_shard_{SHARD_ID}/policy_ft/` — Stockfish-move policy rows

Run **several copies in parallel** with different `SHARD_ID`. CPU is enough (SF dominates).

## Upload / attach

- Current cycle checkpoint (`unified_phase_b_best.pt` or `unified_sp_cycleN.pt`)
- `policy_transformer/vocab1.pkl`
- Stockfish Linux binary (e.g. dataset `stockfish-16-linux`)

Download `/kaggle/working/sp_shard_{SHARD_ID}/` after the run. Fuse locally with `merge_ft_shards.py`.


In [ ]:
# === CONFIG (change SHARD_ID per parallel session) ===
SHARD_ID = 0
GAMES = 200
CORES = 4
SF_DEPTH = 12
P_OPTIMAL = 0.7
ERR_THRESH = 0.15
TOP_K = 5
T_SF = 60.0          # MultiPV temperature in centipawns
MULTIPV = 4
MAX_PLIES = 160
DEVICE = "cpu"       # collector: CPU. Set "cuda" only if this session has a GPU to itself.


## Install deps (Kaggle has no python-chess by default)


In [ ]:
!pip install -q chess
import chess
print("chess ok", chess.__version__)


## Write modules


In [ ]:
%%writefile board_utils.py
"""Board / rights tokenization and STM mirroring for unified_mcts."""
from __future__ import annotations

import math
from typing import Optional, Tuple

import chess
import numpy as np

PIECE_MAP = {
    None: 0,
    "P": 1, "N": 2, "B": 3, "R": 4, "Q": 5, "K": 6,
    "p": 7, "n": 8, "b": 9, "r": 10, "q": 11, "k": 12,
}

RIGHTS_DIM = 8
CP_SCALE = 400.0


def tanh_cp(cp: float) -> float:
    return math.tanh(cp / CP_SCALE)


def stm_value_from_white_pov(white_pov: float, mirrored: bool) -> float:
    """
    Convert absolute White-POV score to STM-white label space.

    After encode_board_stm_white, the board is always White-to-move in the
    encoded view. Labels must be advantage for that STM (matches Lichess
    mega / chess-ai-model §4.2). White-POV on the unmirrored board equals
    STM iff not mirrored; if mirrored, negate.
    """
    return -white_pov if mirrored else white_pov


def board_to_piece_tokens(board: chess.Board) -> np.ndarray:
    """64 uint8 tokens, rank 7→0, file 0→7 (matches value_transformer)."""
    tokens = np.zeros(64, dtype=np.uint8)
    idx = 0
    for rank in range(7, -1, -1):
        for file in range(8):
            piece = board.piece_at(chess.square(file, rank))
            tokens[idx] = PIECE_MAP[piece.symbol()] if piece else 0
            idx += 1
    return tokens


def rights_vector(board: chess.Board) -> np.ndarray:
    """
    K=8 rights vector for the given board (no mirroring applied here).
    Indices: W_OO, W_OOO, B_OO, B_OOO, ep_present, ep_file, 0, 0
    """
    r = np.zeros(RIGHTS_DIM, dtype=np.uint8)
    r[0] = int(board.has_kingside_castling_rights(chess.WHITE))
    r[1] = int(board.has_queenside_castling_rights(chess.WHITE))
    r[2] = int(board.has_kingside_castling_rights(chess.BLACK))
    r[3] = int(board.has_queenside_castling_rights(chess.BLACK))
    if board.ep_square is not None:
        r[4] = 1
        r[5] = chess.square_file(board.ep_square)
    return r


def mirror_rights_for_black_stm(r: np.ndarray) -> np.ndarray:
    """After vertical color-mirror, swap white/black castling; EP file unchanged."""
    out = r.copy()
    out[0], out[2] = r[2], r[0]
    out[1], out[3] = r[3], r[1]
    return out


def encode_board_stm_white(board: chess.Board) -> Tuple[np.ndarray, np.ndarray, bool]:
    """
    Return (piece_tokens, rights, mirrored) with side-to-move always White.
    If Black to move: mirror board + swap castling colors.
    """
    mirrored = board.turn == chess.BLACK
    if mirrored:
        view = board.mirror()
        pieces = board_to_piece_tokens(view)
        rights = mirror_rights_for_black_stm(rights_vector(board))
        # EP file after vertical mirror is the same file index
        if board.ep_square is not None:
            rights[4] = 1
            rights[5] = chess.square_file(board.ep_square)
        else:
            rights[4] = 0
            rights[5] = 0
    else:
        pieces = board_to_piece_tokens(board)
        rights = rights_vector(board)
    return pieces, rights, mirrored


def castle_ep_ids(rights: np.ndarray) -> Tuple[int, int]:
    castle_id = int(rights[0] + 2 * rights[1] + 4 * rights[2] + 8 * rights[3])
    ep_id = 0 if int(rights[4]) == 0 else 1 + int(rights[5])
    return castle_id, ep_id


def get_uci4_token(uci: str) -> str:
    return uci[:4]


def mirror_uci_string(uci: str) -> str:
    if not uci or len(uci) < 4:
        return uci
    new_from = str(9 - int(uci[1]))
    new_to = str(9 - int(uci[3]))
    promo = uci[4:] if len(uci) > 4 else ""
    return f"{uci[0]}{new_from}{uci[2]}{new_to}{promo}"


def pad_history(token_ids: list, block_size: int, pad_id: int) -> np.ndarray:
    """Left-pad to block_size."""
    if len(token_ids) > block_size:
        token_ids = token_ids[-block_size:]
    out = np.full(block_size, pad_id, dtype=np.uint16)
    if token_ids:
        out[-len(token_ids) :] = np.asarray(token_ids, dtype=np.uint16)
    return out


In [ ]:
%%writefile model.py
"""
JointChessNet: board encoder (CLS) + FiLM-conditioned UCI decoder + policy/value heads.
"""
from __future__ import annotations

import math
from typing import Dict, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F

def _apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    x_even = x[..., ::2]
    x_odd = x[..., 1::2]
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    out_even = x_even * cos - x_odd * sin
    out_odd = x_even * sin + x_odd * cos
    return torch.stack((out_even, out_odd), dim=-1).flatten(-2)


class BoardSelfAttention(nn.Module):
    def __init__(self, n_embd: int, n_head: int, dropout: float):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.n_embd = n_embd
        self.dropout = dropout
        self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        self.flash = hasattr(F, "scaled_dot_product_attention")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        if self.flash:
            y = F.scaled_dot_product_attention(
                q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=False
            )
        else:
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y))


class FeedForward(nn.Module):
    def __init__(self, n_embd: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class BoardEncoderBlock(nn.Module):
    def __init__(self, n_embd: int, n_head: int, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.sa = BoardSelfAttention(n_embd, n_head, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ff = FeedForward(n_embd, dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.sa(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


class BoardEncoder(nn.Module):
    def __init__(
        self,
        n_embd: int = 384,
        n_head: int = 6,
        n_layer: int = 6,
        dropout: float = 0.1,
        n_rights_tokens: int = 2,
    ):
        super().__init__()
        self.n_embd = n_embd
        self.piece_emb = nn.Embedding(13, n_embd)
        self.rank_emb = nn.Embedding(8, n_embd)
        self.file_emb = nn.Embedding(8, n_embd)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, n_embd))
        self.castle_emb = nn.Embedding(16, n_embd)
        self.ep_emb = nn.Embedding(9, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(
            [BoardEncoderBlock(n_embd, n_head, dropout) for _ in range(n_layer)]
        )
        self.ln_f = nn.LayerNorm(n_embd)
        nn.init.normal_(self.cls_token, mean=0.0, std=0.02)

    def forward(self, board: torch.Tensor, rights: torch.Tensor) -> torch.Tensor:
        """
        board: [B, 64] long
        rights: [B, 8] long/byte
        returns CLS summary [B, d_b]
        """
        B = board.size(0)
        device = board.device
        ranks = torch.arange(64, device=device) // 8
        files = torch.arange(64, device=device) % 8
        ranks = ranks.unsqueeze(0).expand(B, -1)
        files = files.unsqueeze(0).expand(B, -1)

        e = self.piece_emb(board) + self.rank_emb(ranks) + self.file_emb(files)

        castle_ids = (
            rights[:, 0]
            + 2 * rights[:, 1]
            + 4 * rights[:, 2]
            + 8 * rights[:, 3]
        ).long()
        ep_ids = torch.where(
            rights[:, 4] > 0, 1 + rights[:, 5].long(), torch.zeros(B, device=device, dtype=torch.long)
        )

        cls = self.cls_token.expand(B, -1, -1)
        castle_tok = self.castle_emb(castle_ids).unsqueeze(1)
        ep_tok = self.ep_emb(ep_ids).unsqueeze(1)
        x = torch.cat([cls, e, castle_tok, ep_tok], dim=1)
        x = self.drop(x)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        return x[:, 0]


class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd: int, n_head: int, block_size: int, dropout: float):
        super().__init__()
        assert n_embd % n_head == 0
        self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        self.n_head = n_head
        self.n_embd = n_embd
        self.dropout = dropout
        head_dim = n_embd // n_head
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (10000.0 ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        self.register_buffer("_rope_inv_freq", inv_freq, persistent=False)
        self.register_buffer("_rope_cos_cached", torch.empty(0), persistent=False)
        self.register_buffer("_rope_sin_cached", torch.empty(0), persistent=False)
        self._rope_seq_len_cached = 0
        self._rope_cache_dtype = None
        self._rope_cache_device = None
        self.flash = hasattr(F, "scaled_dot_product_attention")
        if not self.flash:
            self.register_buffer(
                "bias", torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size)
            )

    def _rope_cos_sin(self, seq_len, device, dtype):
        if (
            self._rope_seq_len_cached >= seq_len
            and self._rope_cache_device == device
            and self._rope_cache_dtype == dtype
            and self._rope_cos_cached.numel() != 0
        ):
            return self._rope_cos_cached[:seq_len], self._rope_sin_cached[:seq_len]
        t = torch.arange(seq_len, device=device, dtype=torch.float32)
        freqs = torch.outer(t, self._rope_inv_freq.to(device=device))
        cos = freqs.cos().to(dtype=dtype)
        sin = freqs.sin().to(dtype=dtype)
        self._rope_cos_cached = cos
        self._rope_sin_cached = sin
        self._rope_seq_len_cached = seq_len
        self._rope_cache_device = device
        self._rope_cache_dtype = dtype
        return cos, sin

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        cos, sin = self._rope_cos_sin(T, q.device, q.dtype)
        q = _apply_rope(q, cos, sin)
        k = _apply_rope(k, cos, sin)
        if self.flash:
            y = F.scaled_dot_product_attention(
                q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=True
            )
        else:
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float("-inf"))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y))


class FiLM(nn.Module):
    def __init__(self, d_c: int, d: int):
        super().__init__()
        self.proj = nn.Linear(d_c, 2 * d)

    def forward(self, h: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
        # h: [B,T,d], c: [B,d_c]
        gb = self.proj(c)
        gamma, beta = gb.chunk(2, dim=-1)
        gamma = gamma.unsqueeze(1)
        beta = beta.unsqueeze(1)
        return (1.0 + gamma) * h + beta


class DecoderBlock(nn.Module):
    def __init__(self, n_embd: int, n_head: int, block_size: int, d_c: int, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.film = FiLM(d_c, n_embd)
        self.sa = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ff = FeedForward(n_embd, dropout)

    def forward(self, x: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
        h = self.film(self.ln1(x), c)
        x = x + self.sa(h)
        x = x + self.ff(self.ln2(x))
        return x


class ValueHead(nn.Module):
    def __init__(self, n_embd: int, dropout: float):
        super().__init__()
        self.drop = nn.Dropout(dropout)
        self.net = nn.Sequential(
            nn.Linear(n_embd, n_embd // 2),
            nn.GELU(),
            nn.Linear(n_embd // 2, 1),
            nn.Tanh(),
        )

    def forward(self, s: torch.Tensor) -> torch.Tensor:
        return self.net(self.drop(s)).squeeze(-1)


class JointChessNet(nn.Module):
    def __init__(
        self,
        vocab_size: int = 1793,
        block_size: int = 128,
        d_b: int = 384,
        n_head_b: int = 6,
        n_layer_b: int = 6,
        d: int = 500,
        n_head: int = 10,
        n_layer: int = 8,
        d_c: int | None = None,  # defaults to decoder width d
        dropout: float = 0.1,
    ):
        super().__init__()
        if d_c is None:
            d_c = d
        if d_c != d:
            raise ValueError(f"FiLM condition dim d_c={d_c} must match decoder embedding d={d}")
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.d_c = d_c
        self.encoder = BoardEncoder(d_b, n_head_b, n_layer_b, dropout)
        self.cond_proj = nn.Linear(d_b, d_c)  # board summary → decoder width
        self.token_emb = nn.Embedding(vocab_size, d)
        self.blocks = nn.ModuleList(
            [DecoderBlock(d, n_head, block_size, d_c, dropout) for _ in range(n_layer)]
        )
        self.ln_f = nn.LayerNorm(d)
        self.lm_head = nn.Linear(d, vocab_size)
        self.value_head = ValueHead(d_b, dropout)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def encode_board(self, board: torch.Tensor, rights: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        s = self.encoder(board.long(), rights.long())
        c = self.cond_proj(s)
        return s, c

    def forward_policy(
        self,
        hist: torch.Tensor,
        cond: torch.Tensor,
        targets: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """
        hist: [B, T]
        cond: [B, d_c]
        targets: [B] next-move ids (single-target) or None for logits only
        """
        B, T = hist.shape
        if T > self.block_size:
            hist = hist[:, -self.block_size :]
            T = self.block_size
        x = self.token_emb(hist.long())
        for block in self.blocks:
            x = block(x, cond)
        x = self.ln_f(x)
        logits = self.lm_head(x[:, -1, :])  # [B, V]
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits, targets.long())
        return logits, loss

    def forward_value(self, summary: torch.Tensor, y_v: Optional[torch.Tensor] = None):
        v = self.value_head(summary)
        loss = None
        if y_v is not None:
            loss = F.mse_loss(v, y_v.float())
        return v, loss

    def forward(
        self,
        hist: torch.Tensor,
        board: torch.Tensor,
        rights: torch.Tensor,
        y_pi: Optional[torch.Tensor] = None,
        y_v: Optional[torch.Tensor] = None,
        compute_policy: bool = True,
        compute_value: bool = True,
    ) -> Dict[str, Optional[torch.Tensor]]:
        s, c = self.encode_board(board, rights)
        out: Dict[str, Optional[torch.Tensor]] = {
            "summary": s,
            "cond": c,
            "logits": None,
            "v": None,
            "loss_pi": None,
            "loss_v": None,
        }
        if compute_policy:
            logits, loss_pi = self.forward_policy(hist, c, y_pi)
            out["logits"] = logits
            out["loss_pi"] = loss_pi
        if compute_value:
            v, loss_v = self.forward_value(s, y_v)
            out["v"] = v
            out["loss_v"] = loss_v
        return out

    def config_dict(self) -> dict:
        return {
            "vocab_size": self.vocab_size,
            "block_size": self.block_size,
            "d_b": self.encoder.n_embd,
            "n_head_b": self.encoder.blocks[0].sa.n_head,
            "n_layer_b": len(self.encoder.blocks),
            "d": self.token_emb.embedding_dim,
            "n_head": self.blocks[0].sa.n_head,
            "n_layer": len(self.blocks),
            "d_c": self.d_c,
        }

    def encoder_params(self):
        return list(self.encoder.parameters()) + list(self.cond_proj.parameters())

    def value_params(self):
        return list(self.value_head.parameters())

    def decoder_params(self):
        return (
            list(self.token_emb.parameters())
            + list(self.blocks.parameters())
            + list(self.ln_f.parameters())
            + list(self.lm_head.parameters())
        )

    def set_train_mode(self, mode: str):
        """
        Freeze schedules:
          phase_a / ft_value  — train encoder+value; freeze decoder
          phase_b / ft_policy — train decoder; freeze encoder+value
          all                 — everything trainable
        """
        if mode in ("phase_a", "ft_value"):
            set_requires_grad(self.encoder, True)
            set_requires_grad(self.cond_proj, False)
            set_requires_grad(self.value_head, True)
            set_requires_grad(self.token_emb, False)
            set_requires_grad(self.blocks, False)
            set_requires_grad(self.ln_f, False)
            set_requires_grad(self.lm_head, False)
        elif mode in ("phase_b", "ft_policy"):
            set_requires_grad(self.encoder, False)
            set_requires_grad(self.cond_proj, True)
            set_requires_grad(self.value_head, False)
            set_requires_grad(self.token_emb, True)
            set_requires_grad(self.blocks, True)
            set_requires_grad(self.ln_f, True)
            set_requires_grad(self.lm_head, True)
        elif mode == "all":
            for p in self.parameters():
                p.requires_grad = True
        else:
            raise ValueError(f"Unknown train mode: {mode}")


def set_requires_grad(module: nn.Module, flag: bool):
    for p in module.parameters():
        p.requires_grad = flag


def count_parameters(model: nn.Module) -> float:
    return sum(p.numel() for p in model.parameters()) / 1e6


def count_trainable(model: nn.Module) -> float:
    return sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6


In [ ]:
%%writefile dataset.py
"""Memmap datasets for value / policy training and FT mixes."""
from __future__ import annotations

import json
import os
import pickle
from typing import Dict, Iterator, List, Optional, Sequence, Tuple

import chess
import numpy as np
import torch
from torch.utils.data import Dataset, Sampler

from board_utils import (
    encode_board_stm_white,
    get_uci4_token,
    mirror_uci_string,
    pad_history,
)


def _open_memmap(path: str, dtype, shape, mode: str = "r"):
    return np.memmap(path, dtype=dtype, mode=mode, shape=tuple(shape))


class ValueMemmapDataset(Dataset):
    def __init__(self, data_dir: str, split: str = "train", val_ratio: float = 0.1, seed: int = 42):
        meta_path = os.path.join(data_dir, "value_meta.json")
        with open(meta_path) as f:
            self.meta = json.load(f)
        n = int(self.meta["N"])
        shape_x = (n, 64)
        shape_r = (n, int(self.meta.get("K", 8)))
        self.X = _open_memmap(os.path.join(data_dir, "value_X.u8.memmap"), np.uint8, shape_x)
        self.R = _open_memmap(os.path.join(data_dir, "value_R.u8.memmap"), np.uint8, shape_r)
        self.Y = _open_memmap(os.path.join(data_dir, "value_Y.f16.memmap"), np.float16, (n,))

        rng = np.random.RandomState(seed)
        idx = np.arange(n)
        rng.shuffle(idx)
        n_val = int(n * val_ratio)
        if split == "train":
            self.indices = idx[n_val:]
        else:
            self.indices = idx[:n_val]

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, i: int):
        j = int(self.indices[i])
        board = torch.from_numpy(np.array(self.X[j], dtype=np.int64))
        rights = torch.from_numpy(np.array(self.R[j], dtype=np.int64))
        y = torch.tensor(float(self.Y[j]), dtype=torch.float32)
        return board, rights, y


class PolicyMemmapDataset(Dataset):
    """Single corpus memmap. policy_yv is optional (zeros if missing)."""

    def __init__(self, data_dir: str, start: int = 0, end: Optional[int] = None):
        meta_path = os.path.join(data_dir, "policy_meta.json")
        with open(meta_path) as f:
            self.meta = json.load(f)
        n = int(self.meta["N"])
        self.block_size = int(self.meta.get("block_size", 128))
        self.K = int(self.meta.get("K", 8))
        end = n if end is None else end
        self.start = start
        self.end = end
        self.hist = _open_memmap(
            os.path.join(data_dir, "policy_hist.u16.memmap"),
            np.uint16,
            (n, self.block_size),
        )
        self.board = _open_memmap(
            os.path.join(data_dir, "policy_board.u8.memmap"), np.uint8, (n, 64)
        )
        self.rights = _open_memmap(
            os.path.join(data_dir, "policy_R.u8.memmap"), np.uint8, (n, self.K)
        )
        self.y = _open_memmap(
            os.path.join(data_dir, "policy_y.u16.memmap"), np.uint16, (n,)
        )
        yv_path = os.path.join(data_dir, "policy_yv.f16.memmap")
        if os.path.exists(yv_path):
            self.yv = _open_memmap(yv_path, np.float16, (n,))
            self.has_yv = True
        else:
            self.yv = None
            self.has_yv = False

    def __len__(self) -> int:
        return self.end - self.start

    def __getitem__(self, i: int):
        j = self.start + int(i)
        hist = torch.from_numpy(np.array(self.hist[j], dtype=np.int64))
        board = torch.from_numpy(np.array(self.board[j], dtype=np.int64))
        rights = torch.from_numpy(np.array(self.rights[j], dtype=np.int64))
        y = torch.tensor(int(self.y[j]), dtype=torch.long)
        if self.has_yv:
            yv = torch.tensor(float(self.yv[j]), dtype=torch.float32)
        else:
            yv = torch.tensor(0.0, dtype=torch.float32)
        return hist, board, rights, y, yv


class SourceMixSampler(Sampler[int]):
    """Sample global memmap indices with per-source mix weights (no O(N) weight tensor)."""

    def __init__(
        self,
        ranges: Sequence[Tuple[int, int]],
        weights: Sequence[float],
        num_samples: int,
        seed: int = 42,
    ):
        assert len(ranges) == len(weights) and len(ranges) > 0
        self.ranges = [(int(a), int(b)) for a, b in ranges]
        w = np.asarray(weights, dtype=np.float64)
        w = np.clip(w, 0.0, None)
        if w.sum() <= 0:
            w = np.ones_like(w)
        self.probs = w / w.sum()
        self.num_samples = int(num_samples)
        self.rng = np.random.RandomState(seed)

    def __iter__(self) -> Iterator[int]:
        for _ in range(self.num_samples):
            s = int(self.rng.choice(len(self.ranges), p=self.probs))
            start, end = self.ranges[s]
            yield int(self.rng.randint(start, end))

    def __len__(self) -> int:
        return self.num_samples


class MixedPolicyDataset(Dataset):
    """
    Policy corpus with 80/10/10 (or custom) source mix at sample time.
    meta.json sources: [{name, start, end}, ...]
    """

    def __init__(self, data_dir: str, mix: Optional[Dict[str, float]] = None):
        mix = mix or {"full": 0.80, "elite": 0.10, "weak": 0.10}
        meta_path = os.path.join(data_dir, "policy_meta.json")
        with open(meta_path) as f:
            self.meta = json.load(f)
        if self.meta.get("format") == "pt_replay":
            raise ValueError("Use PtReplayPolicyDataset for format=pt_replay")
        self.base = PolicyMemmapDataset(data_dir)
        sources = self.meta.get("sources")
        if not sources:
            sources = [{"name": "full", "start": 0, "end": int(self.meta["N"])}]

        self.ranges: List[Tuple[int, int]] = []
        self.source_weights: List[float] = []
        self.source_names: List[str] = []
        for src in sources:
            name = src["name"]
            w = float(mix.get(name, 0.0))
            start, end = int(src["start"]), int(src["end"])
            if w <= 0 or end <= start:
                continue
            self.ranges.append((start, end))
            self.source_weights.append(w)
            self.source_names.append(name)

        if not self.ranges:
            n = int(self.meta["N"])
            self.ranges = [(0, n)]
            self.source_weights = [1.0]
            self.source_names = ["all"]

        self._len = int(self.meta["N"])

    def __len__(self) -> int:
        return self._len

    def __getitem__(self, j: int):
        j = int(j)
        hist = torch.from_numpy(np.array(self.base.hist[j], dtype=np.int64))
        board = torch.from_numpy(np.array(self.base.board[j], dtype=np.int64))
        rights = torch.from_numpy(np.array(self.base.rights[j], dtype=np.int64))
        y = torch.tensor(int(self.base.y[j]), dtype=torch.long)
        if self.base.has_yv:
            yv = torch.tensor(float(self.base.yv[j]), dtype=torch.float32)
        else:
            yv = torch.tensor(0.0, dtype=torch.float32)
        return hist, board, rights, y, yv

    def make_sampler(self, num_samples: Optional[int] = None, seed: int = 42) -> SourceMixSampler:
        n = num_samples or self._len
        return SourceMixSampler(self.ranges, self.source_weights, n, seed=seed)


def _first_move_legal(uci: str) -> bool:
    b = chess.Board()
    try:
        b.push_uci(uci)
        return True
    except ValueError:
        return False


def _push_uci4(board: chess.Board, uci4: str) -> bool:
    """Push a 4-char UCI token; vocab has no promo piece, so fall back to queen."""
    try:
        board.push_uci(uci4)
        return True
    except ValueError:
        pass
    try:
        board.push_uci(uci4 + "q")
        return True
    except ValueError:
        return False


def replay_policy_row(
    x_stream: np.ndarray,
    y_stream: np.ndarray,
    game_start: int,
    end: int,
    itos: dict,
    pad_id: int,
    block_size: int,
) -> Optional[Tuple[np.ndarray, np.ndarray, np.ndarray, int]]:
    """
    Replay framed tokens x[game_start:end+1], return (hist, board, rights, y_id).
    If the token stream is a Black-win mirror, un-mirror for physical push.
    """
    if end < game_start:
        return None
    framed = [int(x_stream[k]) for k in range(game_start, end + 1)]
    if not framed:
        return None
    u0 = itos.get(framed[0])
    if not u0 or len(u0) < 4:
        return None
    mirrored = not _first_move_legal(get_uci4_token(u0))

    board = chess.Board()
    for tid in framed:
        u = itos.get(tid)
        if not u or len(u) < 4:
            return None
        phys = mirror_uci_string(get_uci4_token(u)) if mirrored else get_uci4_token(u)
        if not _push_uci4(board, phys):
            return None

    y_id = int(y_stream[end])
    if y_id < 0 or y_id == pad_id:
        return None
    pieces, rights, _ = encode_board_stm_white(board)
    hist = pad_history(framed, block_size, pad_id)
    return hist, pieces, rights, y_id


class PtReplayPolicyDataset(Dataset):
    """
    80/10/10 mix over full/elite/weak .pt streams with on-the-fly board replay.
    Requires policy_meta.json with format=pt_replay from build_policy_from_pt.py.
    """

    def __init__(self, data_dir: str, mix: Optional[Dict[str, float]] = None, seed: int = 42):
        mix = mix or {"full": 0.80, "elite": 0.10, "weak": 0.10}
        meta_path = os.path.join(data_dir, "policy_meta.json")
        with open(meta_path) as f:
            self.meta = json.load(f)
        if self.meta.get("format") != "pt_replay":
            raise ValueError("PtReplayPolicyDataset expects format=pt_replay")

        with open(self.meta["vocab"], "rb") as f:
            stoi = pickle.load(f)
        self.itos = {int(i): s for s, i in stoi.items()}
        self.pad_id = int(self.meta["pad_id"])
        self.block_size = int(self.meta.get("block_size", 128))

        self.sources = []
        weights = []
        for src in self.meta["sources"]:
            name = src["name"]
            w = float(mix.get(name, 0.0))
            if w <= 0 or int(src["n_rows"]) <= 0:
                continue
            if src.get("x_memmap"):
                x = _open_memmap(
                    os.path.join(data_dir, src["x_memmap"]),
                    np.int16,
                    (int(src["stream_len"]),),
                )
                y = _open_memmap(
                    os.path.join(data_dir, src["y_memmap"]),
                    np.int16,
                    (int(src["stream_len"]),),
                )
            else:
                x = np.asarray(
                    torch.load(src["x_pt"], map_location="cpu", weights_only=True),
                    dtype=np.int16,
                )
                y = np.asarray(
                    torch.load(src["y_pt"], map_location="cpu", weights_only=True),
                    dtype=np.int16,
                )
            n_rows = int(src["n_rows"])
            ends = _open_memmap(
                os.path.join(data_dir, src["end_memmap"]), np.uint32, (n_rows,)
            )
            starts = _open_memmap(
                os.path.join(data_dir, src["start_memmap"]), np.uint32, (n_rows,)
            )
            self.sources.append(
                {"name": name, "x": x, "y": y, "ends": ends, "starts": starts, "n": n_rows}
            )
            weights.append(w)

        if not self.sources:
            raise ValueError("No usable sources in policy_meta.json")
        w = np.asarray(weights, dtype=np.float64)
        self.source_probs = w / w.sum()
        self.rng = np.random.RandomState(seed)
        self._len = int(self.meta["N"])

    def __len__(self) -> int:
        return self._len

    def __getitem__(self, _i: int):
        s = int(self.rng.choice(len(self.sources), p=self.source_probs))
        src = self.sources[s]
        for _ in range(8):
            k = int(self.rng.randint(0, src["n"]))
            row = replay_policy_row(
                src["x"],
                src["y"],
                int(src["starts"][k]),
                int(src["ends"][k]),
                self.itos,
                self.pad_id,
                self.block_size,
            )
            if row is not None:
                hist, board, rights, y_id = row
                return (
                    torch.from_numpy(np.asarray(hist, dtype=np.int64)),
                    torch.from_numpy(np.asarray(board, dtype=np.int64)),
                    torch.from_numpy(np.asarray(rights, dtype=np.int64)),
                    torch.tensor(y_id, dtype=torch.long),
                    torch.tensor(0.0, dtype=torch.float32),
                )
        hist = pad_history([], self.block_size, self.pad_id)
        board = np.zeros(64, dtype=np.uint8)
        rights = np.zeros(8, dtype=np.uint8)
        return (
            torch.from_numpy(np.asarray(hist, dtype=np.int64)),
            torch.from_numpy(board.astype(np.int64)),
            torch.from_numpy(rights.astype(np.int64)),
            torch.tensor(self.pad_id, dtype=torch.long),
            torch.tensor(0.0, dtype=torch.float32),
        )

    def make_sampler(self, num_samples: Optional[int] = None, seed: int = 42) -> Sampler:
        n = num_samples or self._len
        return SourceMixSampler([(0, n)], [1.0], n, seed=seed)


class MixedValueFTDataset(Dataset):
    """
    Mix self-play value FT data with base Lichess value data.
    Returns (board, rights, y_v). Sample fraction `ft_frac` from FT, rest from base.
    """

    def __init__(
        self,
        ft_dir: str,
        base_dir: Optional[str] = None,
        ft_frac: float = 0.8,
        seed: int = 42,
    ):
        self.ft = ValueMemmapDataset(ft_dir, split="train", val_ratio=0.0, seed=seed)
        self.base = ValueMemmapDataset(base_dir, split="train", val_ratio=0.0, seed=seed) if base_dir else None
        self.ft_frac = float(ft_frac)
        self.rng = np.random.RandomState(seed)
        self._len = max(len(self.ft), 1)
        if self.base is not None:
            self._len = max(self._len, len(self.base))

    def __len__(self) -> int:
        return self._len

    def __getitem__(self, i: int):
        use_ft = self.base is None or (self.rng.rand() < self.ft_frac)
        ds = self.ft if use_ft else self.base
        j = int(self.rng.randint(0, len(ds)))
        return ds[j]


class MixedPolicyFTDataset(Dataset):
    """
    Mix self-play policy FT with base policy memmap.
    Returns (hist, board, rights, y_pi, y_v).
    """

    def __init__(
        self,
        ft_dir: str,
        base_dir: Optional[str] = None,
        ft_frac: float = 0.8,
        seed: int = 42,
    ):
        self.ft = PolicyMemmapDataset(ft_dir)
        self.base = PolicyMemmapDataset(base_dir) if base_dir else None
        self.ft_frac = float(ft_frac)
        self.rng = np.random.RandomState(seed)
        self._len = max(len(self.ft), 1)
        if self.base is not None:
            self._len = max(self._len, len(self.base))

    def __len__(self) -> int:
        return self._len

    def __getitem__(self, i: int):
        use_ft = self.base is None or (self.rng.rand() < self.ft_frac)
        ds = self.ft if use_ft else self.base
        j = int(self.rng.randint(0, len(ds)))
        return ds[j]


class MixedPolicyPtReplayFTDataset(Dataset):
    """
    Mix self-play policy memmap FT rows with a pt_replay base corpus.
    Returns (hist, board, rights, y_pi, y_v) — same as PolicyMemmapDataset.
    """

    def __init__(
        self,
        ft_dir: str,
        base_dir: Optional[str] = None,
        ft_frac: float = 0.7,
        seed: int = 42,
        split: str = "train",
        val_ratio: float = 0.05,
    ):
        self.ft = PolicyMemmapDataset(ft_dir)
        n = len(self.ft)
        rng = np.random.RandomState(seed)
        idx = np.arange(n)
        rng.shuffle(idx)
        n_val = int(n * val_ratio)
        self.ft_indices = idx[:n_val] if split == "val" else idx[n_val:]
        if len(self.ft_indices) == 0:
            self.ft_indices = idx

        self.base = None
        if base_dir:
            meta_path = os.path.join(base_dir, "policy_meta.json")
            with open(meta_path) as f:
                meta = json.load(f)
            if meta.get("format") == "pt_replay":
                self.base = PtReplayPolicyDataset(base_dir, seed=seed + 1)
            else:
                self.base = PolicyMemmapDataset(base_dir)
        self.ft_frac = float(ft_frac)
        self.rng = np.random.RandomState(seed + (0 if split == "train" else 17))
        self._len = max(len(self.ft_indices), 1)
        if self.base is not None:
            self._len = max(self._len, len(self.base))

    def __len__(self) -> int:
        return self._len

    def __getitem__(self, i: int):
        use_ft = self.base is None or (self.rng.rand() < self.ft_frac)
        if use_ft:
            j = int(self.ft_indices[int(self.rng.randint(0, len(self.ft_indices)))])
            return self.ft[j]
        j = int(self.rng.randint(0, len(self.base)))
        return self.base[j]


def write_memmap(path: str, arr: np.ndarray):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    mm = np.memmap(path, dtype=arr.dtype, mode="w+", shape=arr.shape)
    mm[:] = arr
    mm.flush()
    del mm


class StreamingPolicyWriter:
    """Append policy rows to raw memmap files without holding the full corpus in RAM."""

    def __init__(self, out_dir: str, block_size: int, rights_dim: int, flush_every: int = 50_000):
        self.out_dir = out_dir
        self.block_size = block_size
        self.rights_dim = rights_dim
        self.flush_every = flush_every
        os.makedirs(out_dir, exist_ok=True)
        self.paths = {
            "hist": os.path.join(out_dir, "policy_hist.u16.memmap"),
            "board": os.path.join(out_dir, "policy_board.u8.memmap"),
            "rights": os.path.join(out_dir, "policy_R.u8.memmap"),
            "y": os.path.join(out_dir, "policy_y.u16.memmap"),
            "yv": os.path.join(out_dir, "policy_yv.f16.memmap"),
        }
        for p in self.paths.values():
            open(p, "wb").close()
        self.n = 0
        self._hist: List[np.ndarray] = []
        self._board: List[np.ndarray] = []
        self._rights: List[np.ndarray] = []
        self._y: List[np.ndarray] = []
        self._yv: List[np.ndarray] = []

    def add(self, hist, board, rights, y, yv) -> None:
        self._hist.append(np.asarray(hist, dtype=np.uint16))
        self._board.append(np.asarray(board, dtype=np.uint8))
        self._rights.append(np.asarray(rights, dtype=np.uint8))
        self._y.append(np.asarray(y, dtype=np.uint16))
        self._yv.append(np.asarray(yv, dtype=np.float16))
        if len(self._y) >= self.flush_every:
            self.flush()

    def flush(self) -> None:
        if not self._y:
            return
        hist = np.stack(self._hist).astype(np.uint16, copy=False)
        board = np.stack(self._board).astype(np.uint8, copy=False)
        rights = np.stack(self._rights).astype(np.uint8, copy=False)
        y = np.asarray(self._y, dtype=np.uint16)
        yv = np.asarray(self._yv, dtype=np.float16)
        with open(self.paths["hist"], "ab") as f:
            f.write(hist.tobytes())
        with open(self.paths["board"], "ab") as f:
            f.write(board.tobytes())
        with open(self.paths["rights"], "ab") as f:
            f.write(rights.tobytes())
        with open(self.paths["y"], "ab") as f:
            f.write(y.tobytes())
        with open(self.paths["yv"], "ab") as f:
            f.write(yv.tobytes())
        self.n += len(y)
        self._hist.clear()
        self._board.clear()
        self._rights.clear()
        self._y.clear()
        self._yv.clear()

    def close(self) -> int:
        self.flush()
        return self.n


In [ ]:
%%writefile inference.py
"""Inference helpers for JointChessNet (policy priors + value)."""
from __future__ import annotations

import io
import os
import pickle
import sys
from typing import List, Optional, Tuple

import chess
import chess.pgn
import numpy as np
import torch

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from board_utils import (
    encode_board_stm_white,
    get_uci4_token,
    mirror_uci_string,
    pad_history,
)
from model import JointChessNet


class UnifiedInference:
    def __init__(
        self,
        checkpoint_path: str,
        vocab_path: str,
        device: Optional[str] = None,
    ):
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
            if device == "cpu" and torch.backends.mps.is_available():
                device = "mps"
        self.device = device

        with open(vocab_path, "rb") as f:
            self.stoi = pickle.load(f)
        self.itos = {i: s for s, i in self.stoi.items()}
        self.pad_id = int(self.stoi["|"])
        vocab_size = len(self.stoi)

        ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
        cfg = ckpt.get("config", {}) if isinstance(ckpt, dict) else {}
        self.model = JointChessNet(
            vocab_size=cfg.get("vocab_size", vocab_size),
            block_size=cfg.get("block_size", 128),
            d_b=cfg.get("d_b", 384),
            n_head_b=cfg.get("n_head_b", 6),
            n_layer_b=cfg.get("n_layer_b", 6),
            d=cfg.get("d", 500),
            n_head=cfg.get("n_head", 10),
            n_layer=cfg.get("n_layer", 8),
            d_c=cfg.get("d_c", cfg.get("d", 500)),
            dropout=0.0,
        ).to(device)
        state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
        self.model.load_state_dict(state, strict=False)
        self.model.eval()
        self.block_size = self.model.block_size

    def _encode_position(self, board: chess.Board, moves: List[chess.Move]):
        """Encode current board + move history for STM-white inference."""
        should_mirror = board.turn == chess.BLACK
        hist_ids = []
        for m in moves:
            uci = m.uci()
            tok = get_uci4_token(mirror_uci_string(uci) if should_mirror else uci)
            hist_ids.append(self.stoi.get(tok, self.pad_id))
        pieces, rights, _ = encode_board_stm_white(board)
        hist = pad_history(hist_ids, self.block_size, self.pad_id)
        return hist, pieces, rights, should_mirror

    def _history_and_board_from_pgn(self, pgn_string: str):
        pgn_io = io.StringIO(pgn_string)
        game = chess.pgn.read_game(pgn_io)
        board = chess.Board()
        moves: List[chess.Move] = []
        if game is not None:
            for m in game.mainline_moves():
                moves.append(m)
                board.push(m)
        hist, pieces, rights, should_mirror = self._encode_position(board, moves)
        return board, hist, pieces, rights, should_mirror

    @torch.no_grad()
    def predict_policy_board(
        self,
        board: chess.Board,
        moves: Optional[List[chess.Move]] = None,
        top_k: int = 10,
    ) -> List[Tuple[float, str]]:
        """Return [(logit, uci), ...] legal-masked from a live board + move list."""
        moves = moves or []
        legal = list(board.legal_moves)
        if not legal:
            return []
        hist, pieces, rights, should_mirror = self._encode_position(board, moves)
        hist_t = torch.tensor([hist], dtype=torch.long, device=self.device)
        board_t = torch.tensor([pieces], dtype=torch.long, device=self.device)
        rights_t = torch.tensor([rights], dtype=torch.long, device=self.device)
        out = self.model(hist_t, board_t, rights_t, compute_policy=True, compute_value=False)
        logits = out["logits"][0]

        cands = []
        for move in legal:
            uci = move.uci()
            search = mirror_uci_string(uci) if should_mirror else uci
            tok = get_uci4_token(search)
            if tok in self.stoi:
                score = float(logits[self.stoi[tok]].item())
            else:
                score = float("-inf")
            cands.append((score, uci))
        cands.sort(key=lambda x: x[0], reverse=True)
        return cands[:top_k]

    @torch.no_grad()
    def predict_policy(self, pgn_string: str, top_k: int = 10) -> List[Tuple[float, str]]:
        """Return [(logit, uci), ...] legal-masked, sorted descending — MCTS policy_func shape."""
        board, hist, pieces, rights, should_mirror = self._history_and_board_from_pgn(pgn_string)
        # Reuse board path
        moves = []
        temp = chess.Board()
        pgn_io = io.StringIO(pgn_string)
        game = chess.pgn.read_game(pgn_io)
        if game is not None:
            for m in game.mainline_moves():
                moves.append(m)
                temp.push(m)
        return self.predict_policy_board(board, moves, top_k=top_k)

    @torch.no_grad()
    def evaluate_board(self, board: chess.Board) -> float:
        pieces, rights, _ = encode_board_stm_white(board)
        hist = pad_history([], self.block_size, self.pad_id)
        hist_t = torch.tensor([hist], dtype=torch.long, device=self.device)
        board_t = torch.tensor([pieces], dtype=torch.long, device=self.device)
        rights_t = torch.tensor([rights], dtype=torch.long, device=self.device)
        out = self.model(hist_t, board_t, rights_t, compute_policy=False, compute_value=True)
        return float(out["v"][0].item())

    def evaluate_fen(self, fen: str) -> float:
        return self.evaluate_board(chess.Board(fen))

    @torch.no_grad()
    def choose_move(
        self,
        board: chess.Board,
        moves: Optional[List[chess.Move]] = None,
        top_k: int = 5,
        temperature: float = 0.0,
        dirichlet_alpha: float = 0.25,
        dirichlet_eps: float = 0.25,
    ) -> Optional[chess.Move]:
        """Sample or argmax a legal move from policy logits."""
        cands = self.predict_policy_board(board, moves, top_k=top_k)
        if not cands:
            return None
        ucis = [u for _, u in cands]
        logits = np.array([s for s, _ in cands], dtype=np.float64)
        if temperature <= 0:
            return chess.Move.from_uci(ucis[int(np.argmax(logits))])
        logits = logits / max(temperature, 1e-6)
        logits = logits - logits.max()
        probs = np.exp(logits)
        probs = probs / (probs.sum() + 1e-12)
        if dirichlet_eps > 0 and len(probs) > 1:
            noise = np.random.dirichlet([dirichlet_alpha] * len(probs))
            probs = (1 - dirichlet_eps) * probs + dirichlet_eps * noise
            probs = probs / (probs.sum() + 1e-12)
        pick = np.random.choice(ucis, p=probs)
        return chess.Move.from_uci(pick)


In [ ]:
%%writefile self_play_sf.py
#!/usr/bin/env python3
"""
Collect value + policy FT shards: unified vs Stockfish.

Unified: policy top-k, then Bernoulli P_OPTIMAL value-rerank vs temperature sample.
Stockfish: MultiPV softmax(-cp_loss / T_SF) — mostly optimal with temperature.

Value rows: |v_θ − y_SF| ≥ err_thresh on unified candidate children (+ random floor).
Policy rows: every SF ply (STM-white, single-target) plus optional SF-best at error boards.

Writes:
  <out>/value_ft/{value_X,value_R,value_Y,value_meta.json}
  <out>/policy_ft/{policy_hist,policy_board,policy_R,policy_y,policy_yv,policy_meta.json}

Example:
  python self_play_sf.py \\
    --ckpt ../ckpts/unified_phase_b_best.pt \\
    --vocab ../policy_transformer/vocab1.pkl \\
    --stockfish /opt/homebrew/bin/stockfish \\
    --out ./data_export/sp_shard_0 \\
    --shard-id 0 --games 200 --cores 4
"""
from __future__ import annotations

import argparse
import json
import os
import pickle
import random
import sys
from typing import Any, Dict, List, Optional, Tuple

import chess
import chess.engine
import numpy as np
import torch.multiprocessing as mp
from tqdm import tqdm

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from board_utils import (
    RIGHTS_DIM,
    encode_board_stm_white,
    get_uci4_token,
    mirror_uci_string,
    pad_history,
    stm_value_from_white_pov,
    tanh_cp,
)
from dataset import write_memmap
from inference import UnifiedInference

# Populated in worker_init (spawn-safe).
_G: Dict[str, Any] = {}


def eval_sf_white_pov(board: chess.Board, engine: chess.engine.SimpleEngine, depth: int) -> float:
    info = engine.analyse(board, chess.engine.Limit(depth=depth))
    score = info["score"].white()
    if score.is_mate():
        m = score.mate()
        cp = 10000.0 if m and m > 0 else -10000.0
    else:
        cp = float(score.score(mate_score=10000) or 0)
    return tanh_cp(cp)


def eval_sf_stm(board: chess.Board, engine: chess.engine.SimpleEngine, depth: int, mirrored: bool) -> float:
    return stm_value_from_white_pov(eval_sf_white_pov(board, engine, depth), mirrored)


def _relative_cp(info: dict) -> float:
    score = info["score"].relative
    if score.is_mate():
        m = score.mate()
        return 10000.0 if m and m > 0 else -10000.0
    return float(score.score(mate_score=10000) or 0)


def sf_sample_move(
    board: chess.Board,
    engine: chess.engine.SimpleEngine,
    depth: int,
    multipv: int,
    t_sf: float,
    rng: np.random.RandomState,
) -> chess.Move:
    """Sample among MultiPV lines with softmax(-cp_loss / T_SF)."""
    infos = engine.analyse(board, chess.engine.Limit(depth=depth), multipv=multipv)
    if not isinstance(infos, list):
        infos = [infos]
    moves, cps = [], []
    for inf in infos:
        pv = inf.get("pv") or []
        if not pv:
            continue
        mv = pv[0]
        if mv not in board.legal_moves:
            continue
        moves.append(mv)
        cps.append(_relative_cp(inf))
    if not moves:
        result = engine.play(board, chess.engine.Limit(depth=depth))
        return result.move
    cps = np.asarray(cps, dtype=np.float64)
    best = cps.max()
    loss = best - cps
    logits = -loss / max(float(t_sf), 1e-6)
    logits = logits - logits.max()
    probs = np.exp(logits)
    probs = probs / (probs.sum() + 1e-12)
    return moves[int(rng.choice(len(moves), p=probs))]


def history_ids(prior: List[chess.Move], mirrored: bool, stoi: dict, pad_id: int) -> List[int]:
    ids = []
    for m in prior:
        tok = get_uci4_token(mirror_uci_string(m.uci()) if mirrored else m.uci())
        ids.append(int(stoi[tok]) if tok in stoi else pad_id)
    return ids


def policy_row(
    board: chess.Board,
    prior: List[chess.Move],
    move: chess.Move,
    stoi: dict,
    pad_id: int,
    block_size: int,
) -> Optional[dict]:
    pieces, rights, mirrored = encode_board_stm_white(board)
    token = get_uci4_token(mirror_uci_string(move.uci()) if mirrored else move.uci())
    if token not in stoi:
        return None
    hist = pad_history(history_ids(prior, mirrored, stoi, pad_id), block_size, pad_id)
    return {
        "hist": hist,
        "board": pieces,
        "rights": rights,
        "y": np.uint16(stoi[token]),
    }


def unified_pick(
    inf: UnifiedInference,
    board: chess.Board,
    moves: List[chess.Move],
    cands: List[Tuple[float, str]],
    p_optimal: float,
    temperature: float,
    rng: np.random.RandomState,
) -> chess.Move:
    if not cands:
        return list(board.legal_moves)[0]
    if rng.rand() < p_optimal:
        best_uci, best_s = cands[0][1], -1e9
        for _, uci in cands:
            mv = chess.Move.from_uci(uci)
            if mv not in board.legal_moves:
                continue
            board.push(mv)
            s = -float(inf.evaluate_board(board))
            board.pop()
            if s > best_s:
                best_s, best_uci = s, uci
        return chess.Move.from_uci(best_uci)
    mv = inf.choose_move(
        board,
        moves,
        top_k=len(cands),
        temperature=max(temperature, 0.3),
        dirichlet_eps=0.0,
    )
    return mv or chess.Move.from_uci(cands[0][1])


def play_one_game(game_id: int) -> dict:
    inf: UnifiedInference = _G["inf"]
    engine: chess.engine.SimpleEngine = _G["engine"]
    stoi = _G["stoi"]
    pad_id = _G["pad_id"]
    cfg = _G["cfg"]
    rng = np.random.RandomState(int(cfg["seed"]) + int(game_id) * 10007)

    model_is_white = (game_id + int(cfg["shard_id"])) % 2 == 0
    board = chess.Board()
    moves: List[chess.Move] = []
    value_rows: List[Tuple[np.ndarray, np.ndarray, float]] = []
    policy_rows: List[dict] = []

    for _ply in range(int(cfg["max_plies"])):
        if board.is_game_over():
            break
        white_to_move = board.turn == chess.WHITE
        model_turn = (white_to_move and model_is_white) or ((not white_to_move) and (not model_is_white))

        if model_turn:
            cands = inf.predict_policy_board(board, moves, top_k=int(cfg["top_k"]))
            if not cands:
                break
            mined_here = False
            for _, uci in cands:
                mv = chess.Move.from_uci(uci)
                if mv not in board.legal_moves:
                    continue
                board.push(mv)
                pieces, rights, mirrored = encode_board_stm_white(board)
                v_theta = float(inf.evaluate_board(board))
                try:
                    y_sf = float(eval_sf_stm(board, engine, int(cfg["sf_depth"]), mirrored))
                except Exception:
                    y_sf = 0.0
                err = abs(v_theta - y_sf)
                keep = err >= float(cfg["err_thresh"]) or rng.rand() < float(cfg["floor_frac"])
                if keep:
                    value_rows.append((pieces, rights, y_sf))
                    mined_here = mined_here or err >= float(cfg["err_thresh"])
                board.pop()

            if cfg["policy_from_errors"] and mined_here:
                try:
                    sf_best = engine.play(board, chess.engine.Limit(depth=int(cfg["sf_depth"]))).move
                    if sf_best is not None:
                        row = policy_row(board, moves, sf_best, stoi, pad_id, int(cfg["block_size"]))
                        if row:
                            policy_rows.append(row)
                except Exception:
                    pass

            mv = unified_pick(
                inf, board, moves, cands, float(cfg["p_optimal"]), float(cfg["temperature"]), rng
            )
        else:
            try:
                mv = sf_sample_move(
                    board,
                    engine,
                    int(cfg["sf_depth"]),
                    int(cfg["multipv"]),
                    float(cfg["t_sf"]),
                    rng,
                )
            except Exception:
                break
            if mv is None:
                break
            row = policy_row(board, moves, mv, stoi, pad_id, int(cfg["block_size"]))
            if row:
                policy_rows.append(row)

        if mv not in board.legal_moves:
            break
        board.push(mv)
        moves.append(mv)

    outcome = board.outcome()
    if outcome is None or outcome.winner is None:
        result = "draw"
    elif (outcome.winner == chess.WHITE) == model_is_white:
        result = "model"
    else:
        result = "sf"
    return {
        "value": value_rows,
        "policy": policy_rows,
        "result": result,
        "model_white": model_is_white,
        "plies": len(moves),
    }


def worker_init(ckpt: str, vocab: str, stockfish: str, cfg: dict, device: str):
    inf = UnifiedInference(ckpt, vocab, device=device)
    engine = chess.engine.SimpleEngine.popen_uci(stockfish)
    engine.configure({"Threads": 1})
    if cfg.get("sf_skill") is not None:
        try:
            engine.configure({"Skill Level": int(cfg["sf_skill"])})
        except Exception:
            pass
    with open(vocab, "rb") as f:
        stoi = pickle.load(f)
    _G["inf"] = inf
    _G["engine"] = engine
    _G["stoi"] = stoi
    _G["pad_id"] = int(stoi["|"])
    _G["cfg"] = cfg
    import atexit
    atexit.register(lambda: engine.quit() if engine else None)


def worker_play(game_id: int) -> dict:
    try:
        return play_one_game(game_id)
    except Exception as e:
        return {"value": [], "policy": [], "result": "error", "error": str(e), "plies": 0}


def dump_value(out_dir: str, rows: List[Tuple[np.ndarray, np.ndarray, float]], meta_extra: dict):
    os.makedirs(out_dir, exist_ok=True)
    n = len(rows)
    if n == 0:
        raise SystemExit(f"No value rows collected → {out_dir}")
    X = np.stack([r[0] for r in rows]).astype(np.uint8)
    R = np.stack([r[1] for r in rows]).astype(np.uint8)
    Y = np.asarray([r[2] for r in rows], dtype=np.float16)
    write_memmap(os.path.join(out_dir, "value_X.u8.memmap"), X)
    write_memmap(os.path.join(out_dir, "value_R.u8.memmap"), R)
    write_memmap(os.path.join(out_dir, "value_Y.f16.memmap"), Y)
    meta = {
        "N": n,
        "K": RIGHTS_DIM,
        "y_space": "tanh(cp/400)",
        "source": "self_play_sf",
        **meta_extra,
    }
    with open(os.path.join(out_dir, "value_meta.json"), "w") as f:
        json.dump(meta, f, indent=2)
    print(f"value_ft N={n} → {out_dir}")


def dump_policy(out_dir: str, rows: List[dict], vocab: str, pad_id: int, meta_extra: dict):
    os.makedirs(out_dir, exist_ok=True)
    n = len(rows)
    if n == 0:
        raise SystemExit(f"No policy rows collected → {out_dir}")
    hist = np.stack([r["hist"] for r in rows]).astype(np.uint16)
    board = np.stack([r["board"] for r in rows]).astype(np.uint8)
    rights = np.stack([r["rights"] for r in rows]).astype(np.uint8)
    y = np.asarray([r["y"] for r in rows], dtype=np.uint16)
    yv = np.zeros(n, dtype=np.float16)
    write_memmap(os.path.join(out_dir, "policy_hist.u16.memmap"), hist)
    write_memmap(os.path.join(out_dir, "policy_board.u8.memmap"), board)
    write_memmap(os.path.join(out_dir, "policy_R.u8.memmap"), rights)
    write_memmap(os.path.join(out_dir, "policy_y.u16.memmap"), y)
    write_memmap(os.path.join(out_dir, "policy_yv.f16.memmap"), yv)
    meta = {
        "N": n,
        "K": RIGHTS_DIM,
        "block_size": int(hist.shape[1]),
        "vocab": os.path.abspath(vocab),
        "pad_id": pad_id,
        "has_policy_yv": True,
        "sf_labeled": True,
        "source": "self_play_sf",
        "sources": [{"name": "self_play", "start": 0, "end": n}],
        **meta_extra,
    }
    with open(os.path.join(out_dir, "policy_meta.json"), "w") as f:
        json.dump(meta, f, indent=2)
    print(f"policy_ft N={n} → {out_dir}")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--vocab", required=True)
    ap.add_argument("--stockfish", required=True)
    ap.add_argument("--out", required=True, help="Shard root (writes value_ft/ and policy_ft/)")
    ap.add_argument("--shard-id", type=int, default=0)
    ap.add_argument("--seed", type=int, default=None, help="Default 1000+shard_id")
    ap.add_argument("--games", type=int, default=200)
    ap.add_argument("--cores", type=int, default=4)
    ap.add_argument("--max-plies", type=int, default=160)
    ap.add_argument("--sf-depth", type=int, default=12)
    ap.add_argument("--sf-skill", type=int, default=None, help="Optional UCI Skill Level (0-20)")
    ap.add_argument("--multipv", type=int, default=4)
    ap.add_argument("--t-sf", type=float, default=60.0, help="SF MultiPV temperature in cp")
    ap.add_argument("--top-k", type=int, default=5)
    ap.add_argument("--p-optimal", type=float, default=0.7)
    ap.add_argument("--temperature", type=float, default=0.8, help="Unified explore temperature")
    ap.add_argument("--err-thresh", type=float, default=0.15)
    ap.add_argument("--floor-frac", type=float, default=0.10)
    ap.add_argument("--policy-from-errors", action="store_true", default=True)
    ap.add_argument("--no-policy-from-errors", action="store_false", dest="policy_from_errors")
    ap.add_argument("--block-size", type=int, default=128)
    ap.add_argument("--device", default="cpu")
    args = ap.parse_args()

    seed = args.seed if args.seed is not None else 1000 + int(args.shard_id)
    random.seed(seed)
    np.random.seed(seed)

    with open(args.vocab, "rb") as f:
        stoi = pickle.load(f)
    pad_id = int(stoi["|"])

    cfg = {
        "shard_id": int(args.shard_id),
        "seed": int(seed),
        "max_plies": int(args.max_plies),
        "sf_depth": int(args.sf_depth),
        "sf_skill": args.sf_skill,
        "multipv": int(args.multipv),
        "t_sf": float(args.t_sf),
        "top_k": int(args.top_k),
        "p_optimal": float(args.p_optimal),
        "temperature": float(args.temperature),
        "err_thresh": float(args.err_thresh),
        "floor_frac": float(args.floor_frac),
        "policy_from_errors": bool(args.policy_from_errors),
        "block_size": int(args.block_size),
    }

    os.makedirs(args.out, exist_ok=True)
    try:
        mp.set_start_method("spawn", force=True)
    except RuntimeError:
        pass

    value_all: List[Tuple[np.ndarray, np.ndarray, float]] = []
    policy_all: List[dict] = []
    wins_sf = wins_model = draws = errors = 0

    ctx = mp.get_context("spawn")
    with ctx.Pool(
        processes=int(args.cores),
        initializer=worker_init,
        initargs=(args.ckpt, args.vocab, args.stockfish, cfg, args.device),
    ) as pool:
        for res in tqdm(
            pool.imap_unordered(worker_play, range(int(args.games))),
            total=int(args.games),
            desc=f"sp_shard_{args.shard_id}",
        ):
            value_all.extend(res.get("value") or [])
            policy_all.extend(res.get("policy") or [])
            r = res.get("result")
            if r == "sf":
                wins_sf += 1
            elif r == "model":
                wins_model += 1
            elif r == "draw":
                draws += 1
            else:
                errors += 1

    extra = {
        "sf_depth": args.sf_depth,
        "games": args.games,
        "shard_id": args.shard_id,
        "seed": seed,
        "ckpt": os.path.abspath(args.ckpt),
        "wins_sf": wins_sf,
        "wins_model": wins_model,
        "draws": draws,
        "errors": errors,
        "p_optimal": args.p_optimal,
        "err_thresh": args.err_thresh,
        "t_sf": args.t_sf,
        "multipv": args.multipv,
    }
    dump_value(os.path.join(args.out, "value_ft"), value_all, extra)
    dump_policy(os.path.join(args.out, "policy_ft"), policy_all, args.vocab, pad_id, extra)
    print(
        f"Done shard={args.shard_id} games={args.games} "
        f"SF={wins_sf} model={wins_model} draws={draws} errors={errors}"
    )


if __name__ == "__main__":
    main()


## Find checkpoint, vocab, Stockfish


In [ ]:
import os, shutil, time

input_root = "/kaggle/input"
work = "/kaggle/working"
os.makedirs(work, exist_ok=True)

def find_file(names):
    hits = []
    for root, dirs, files in os.walk(input_root):
        for fn in files:
            if fn in names:
                hits.append(os.path.join(root, fn))
    for prefer in names:
        for h in hits:
            if os.path.basename(h) == prefer:
                return h
    return hits[0] if hits else None

CKPT = find_file([
    "unified_sp_cycle10.pt", "unified_sp_cycle9.pt", "unified_sp_cycle8.pt",
    "unified_sp_cycle7.pt", "unified_sp_cycle6.pt", "unified_sp_cycle5.pt",
    "unified_sp_cycle4.pt", "unified_sp_cycle3.pt", "unified_sp_cycle2.pt",
    "unified_sp_cycle1.pt", "unified_sp_cycle0.pt",
    "unified_phase_b_best.pt", "unified_phase_b.pt",
    "unified_phase_b_best.zip", "unified_phase_b.zip",
])
VOCAB = find_file(["vocab1.pkl"])
print("CKPT", CKPT)
print("VOCAB", VOCAB)
assert CKPT, "Upload current cycle checkpoint (unified_phase_b_best.pt or unified_sp_cycleN.pt)"
assert VOCAB, "Upload vocab1.pkl"

# Stockfish: copy from input to working and chmod +x
SF_EXECUTABLE = os.path.join(work, "stockfish_exec")
sf_src = find_file([
    "stockfish-ubuntu-x86-64-avx2", "stockfish", "stockfish-ubuntu-x86-64",
])
if sf_src is None:
    for root, dirs, files in os.walk(input_root):
        for fn in files:
            if "stockfish" in fn.lower() and not fn.endswith((".txt", ".md")):
                sf_src = os.path.join(root, fn)
                break
        if sf_src:
            break
print("SF_SRC", sf_src)
assert sf_src, "Attach a Stockfish Linux binary dataset"
if not (os.path.exists(SF_EXECUTABLE) and os.access(SF_EXECUTABLE, os.X_OK)):
    shutil.copy2(sf_src, SF_EXECUTABLE)
    os.chmod(SF_EXECUTABLE, 0o755)
    time.sleep(0.5)
print("SF_EXECUTABLE", SF_EXECUTABLE)
assert os.access(SF_EXECUTABLE, os.X_OK)


## Collect shard


In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "chess"])

import os, sys
sys.path.insert(0, "/kaggle/working")

from self_play_sf import main as _unused  # noqa: F401 — ensure imports resolve
import self_play_sf as sp

# Drive via argv so the CLI stays the source of truth
sys.argv = [
    "self_play_sf.py",
    "--ckpt", CKPT,
    "--vocab", VOCAB,
    "--stockfish", SF_EXECUTABLE,
    "--out", f"/kaggle/working/sp_shard_{SHARD_ID}",
    "--shard-id", str(SHARD_ID),
    "--games", str(GAMES),
    "--cores", str(CORES),
    "--sf-depth", str(SF_DEPTH),
    "--p-optimal", str(P_OPTIMAL),
    "--err-thresh", str(ERR_THRESH),
    "--top-k", str(TOP_K),
    "--t-sf", str(T_SF),
    "--multipv", str(MULTIPV),
    "--max-plies", str(MAX_PLIES),
    "--device", DEVICE,
    "--policy-from-errors",
]
sp.main()
print("shard root:", f"/kaggle/working/sp_shard_{SHARD_ID}")
print("Download that folder (value_ft/ + policy_ft/). Fuse locally with merge_ft_shards.py")
